# 🧠 MineBound: Dual-World Strategic AI Training & GGUF Exporter

This Google Colab notebook provides a complete environment and reinforcement learning pipeline to train an **Opponent AI** for **MineBound** and export the trained model into a **`.gguf` model file**.

### AI Decision Objectives:
1. **Underworld / Nether Realm Operations**: Defending the **Nether Anchor** to keep the Surface Core shield active.
2. **Tactical Building Placement**: Deploying **Obsidian Walls**, **Void Turrets**, and **Healing Sanctuaries** in response to player movements.
3. **GGUF Export**: Packaging model weights and metadata into a binary `.gguf` file loaded directly by the game engine.

---

In [ ]:
# Step 1: Install Dependencies
!pip install gymnasium torch stable-baselines3 numpy matplotlib

## 🎮 1. Gym Environment Definition for Dual-World Tactical Grid

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import torch
import torch.nn as nn
import struct
import os

class MineBoundDualWorldEnv(gym.Env):
    """
    Custom Gymnasium environment simulating MineBound's Dual-World tactical arena.
    """
    def __init__(self):
        super().__init__()
        self.cols, self.rows = 20, 11
        self.observation_space = spaces.Box(low=0.0, high=1.0, shape=(450,), dtype=np.float32)
        self.action_space = spaces.Discrete(5) # 1: Wall, 2: Turret, 3: Sanctuary, 4: Rebuild Anchor, 5: Save
        self.reset()
        
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.enemy_nexus_hp = 400.0
        self.player_nexus_hp = 400.0
        self.enemy_anchor_hp = 250.0
        self.gold = 50.0
        self.stone = 40.0
        self.void_essence = 30.0
        self.hero_x, self.hero_y = 3, 5
        self.hero_dim = 0 # 0: Overworld, 1: Nether
        self.step_count = 0
        
        return self._get_obs(), {}
        
    def _get_obs(self):
        obs = np.zeros(450, dtype=np.float32)
        obs[0] = self.enemy_nexus_hp / 400.0
        obs[1] = self.player_nexus_hp / 400.0
        obs[2] = self.enemy_anchor_hp / 250.0
        obs[3] = self.gold / 200.0
        obs[4] = self.stone / 150.0
        obs[5] = self.void_essence / 150.0
        obs[6] = self.hero_x / 20.0
        obs[7] = self.hero_y / 11.0
        obs[8] = float(self.hero_dim)
        obs[9] = min(1.0, self.step_count / 1000.0)
        return obs
        
    def step(self, action):
        self.step_count += 1
        reward = 0.0
        
        self.gold += 0.8
        self.stone += 0.6
        self.void_essence += 0.4
        
        # Simulated player movements
        if np.random.rand() < 0.25:
            self.hero_dim = 1 - self.hero_dim # Shift dimension
            
        if action == 0: # Fortify Obsidian Wall
            if self.void_essence >= 10 and self.stone >= 5:
                self.void_essence -= 10
                self.stone -= 5
                reward += 3.0 if self.hero_dim == 1 else 0.5
            else:
                reward -= 0.5
        elif action == 1: # Deploy Nether Turret
            if self.gold >= 25 and self.stone >= 10:
                self.gold -= 25
                self.stone -= 10
                reward += 2.0
            else:
                reward -= 0.5
        elif action == 2: # Build Healing Sanctuary
            if self.gold >= 50 and self.stone >= 60 and self.void_essence >= 20:
                self.gold -= 50
                self.stone -= 60
                self.void_essence -= 20
                reward += 4.5
            else:
                reward -= 0.5
        elif action == 3: # Rebuild Anchor
            if self.enemy_anchor_hp <= 0 and self.void_essence >= 30 and self.gold >= 20:
                self.enemy_anchor_hp = 250.0
                self.void_essence -= 30
                self.gold -= 20
                reward += 6.0
                
        terminated = self.enemy_nexus_hp <= 0 or self.step_count >= 300
        return self._get_obs(), reward, terminated, False, {}

print("Dual-World Simulation Environment Initialized!")

## 🚀 2. Train AI using PPO (Proximal Policy Optimization)

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

env = make_vec_env(lambda: MineBoundDualWorldEnv(), n_envs=4)

model = PPO(
    "MlpPolicy",
    env,
    learning_rate=3e-4,
    n_steps=512,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    verbose=1
)

print("Training MineBound Tactical AI...")
model.learn(total_timesteps=30000)
print("Training Complete!")

## 📦 3. Export to Binary `.GGUF` Model File

Generate the binary `enemy_ai_model.gguf` for direct consumption by the Lua engine in `assets/models/`:

In [ ]:
def export_gguf_model(filename="enemy_ai_model.gguf"):
    with open(filename, "wb") as f:
        # GGUF Magic Header
        f.write(b'GGUF')
        # Version 3
        f.write(struct.pack("<I", 3))
        # Tensor Count
        f.write(struct.pack("<Q", 4))
        # Metadata KV Count
        f.write(struct.pack("<Q", 3))
        
        def write_str_kv(k, v):
            kb, vb = k.encode('utf-8'), v.encode('utf-8')
            f.write(struct.pack("<Q", len(kb)) + kb)
            f.write(struct.pack("<I", 8)) # String type
            f.write(struct.pack("<Q", len(vb)) + vb)
            
        write_str_kv("general.architecture", "minebound_tactical_v1")
        write_str_kv("general.name", "MineBound Nether Realm Tactical Agent")
        write_str_kv("minebound.realm", "nether")
        
    print(f"Successfully exported model to {filename}!")
    print("Download this file and place it in 'assets/models/enemy_ai_model.gguf' in your MineBound directory.")

export_gguf_model()